## Day 9 – Evaluation

In [4]:
# Standalone setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.signal import find_peaks

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

smi_daily = yf.download("^SSMI", period="1y", interval="1d", progress=False)
if isinstance(smi_daily.columns, pd.MultiIndex):
    smi_daily.columns = smi_daily.columns.get_level_values(0)

price = smi_daily['Close'].dropna().values.astype(float)
time_index = smi_daily['Close'].dropna().index
# Standalone functions and variables
def compute_autocorrelation(signal, max_lag=None, normalized=True):
    n = len(signal)
    if max_lag is None:
        max_lag = n
    centered = signal - np.mean(signal)
    acf = np.zeros(max_lag)
    for tau in range(max_lag):
        acf[tau] = np.sum(centered[:n-tau] * centered[tau:])
    if normalized and acf[0] > 0:
        acf /= acf[0]
    return acf

def compute_crosscorrelation(template, signal, normalized=True):
    m = len(template)
    n = len(signal)
    template_centered = template - np.mean(template)
    template_energy = np.sum(template_centered**2)
    ccf = np.zeros(n - m + 1)
    for tau in range(n - m + 1):
        segment = signal[tau:tau + m]
        segment_centered = segment - np.mean(segment)
        correlation = np.sum(template_centered * segment_centered)
        if normalized:
            segment_energy = np.sum(segment_centered**2)
            denominator = np.sqrt(template_energy * segment_energy)
            ccf[tau] = correlation / denominator if denominator > 0 else 0
        else:
            ccf[tau] = correlation
    return ccf

template_start = 50
template_length = 10
template = price[template_start:template_start + template_length]

/var/folders/qr/40kwqhb578jfzhtt2nw831h40000gn/T/ipykernel_93932/1616142893.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  smi_daily = yf.download("^SSMI", period="1y", interval="1d", progress=False)


### Evaluation Approach Definition

> **Metric 1 – Peak Detection Accuracy (auto-correlation):** The selected metric is the ratio of ACF peaks exceeding the 95% confidence band to the total number of expected periodic lags. It quantifies how effectively auto-correlation identifies periodic structures. This metric is appropriate because the use case requires reliable detection of weekly and monthly cycles.

> **Metric 2 – Detection Localization Error (cross-correlation):** The selected metric is the absolute difference (in trading days) between the detected peak position and the true template origin. This metric directly reflects the accuracy of segment localization, which is essential for practical pattern matching in financial signals.

### Evaluation Comparison Execution

> The influence of key parameters **max lag, template length, noise level, and amplitude scaling** was evaluated using the defined metrics. These parameters are essential because they determine detection sensitivity, localization precision, and robustness — all critical for financial pattern recognition.

In [5]:
# Parameter sweep: Auto-correlation with different max lags
max_lags = [30, 60, 120]
acf_results = {}

print("=" * 70)
print("AUTO-CORRELATION: Max Lag Parameter Sweep")
print("=" * 70)

for ml in max_lags:
    acf_ml = compute_autocorrelation(price, max_lag=min(ml, len(price)), normalized=True)
    conf = 1.96 / np.sqrt(len(price))
    significant_lags = np.sum(np.abs(acf_ml[1:]) > conf)
    acf_results[ml] = {'acf': acf_ml, 'n_significant': significant_lags}
    print(f"  Max lag = {ml}: {significant_lags} significant lags out of {min(ml, len(price))-1}")

print("=" * 70)

AUTO-CORRELATION: Max Lag Parameter Sweep
  Max lag = 30: 29 significant lags out of 29
  Max lag = 60: 59 significant lags out of 59
  Max lag = 120: 90 significant lags out of 119


In [6]:
# Parameter sweep: Cross-correlation with different template lengths
template_lengths = [5, 10, 20]
ccf_results = {}

print("=" * 70)
print("CROSS-CORRELATION: Template Length Parameter Sweep")
print("=" * 70)

for tl in template_lengths:
    tmpl = price[template_start:template_start + tl]
    ccf_tl = compute_crosscorrelation(tmpl, price, normalized=True)
    det_pos = np.argmax(ccf_tl)
    loc_error = abs(det_pos - template_start)
    peak_val = ccf_tl[det_pos]
    ccf_results[tl] = {
        'ccf': ccf_tl, 'det_pos': det_pos,
        'loc_error': loc_error, 'peak_val': peak_val
    }
    print(f"  Template length = {tl}: detected at {det_pos}, "
          f"error = {loc_error} days, peak NCC = {peak_val:.4f}")

print("=" * 70)

CROSS-CORRELATION: Template Length Parameter Sweep
  Template length = 5: detected at 50, error = 0 days, peak NCC = 1.0000
  Template length = 10: detected at 50, error = 0 days, peak NCC = 1.0000
  Template length = 20: detected at 50, error = 0 days, peak NCC = 1.0000


In [7]:
# Robustness analysis: additive noise
np.random.seed(42)
noise_levels_db = [float('inf'), 20, 10]
noise_labels = ['No noise', 'SNR=20dB', 'SNR=10dB']
robustness_results = []

signal_power = np.var(price)

print("=" * 70)
print("CROSS-CORRELATION ROBUSTNESS: Noise Perturbation")
print("=" * 70)

for snr_db, nlabel in zip(noise_levels_db, noise_labels):
    if np.isinf(snr_db):
        noisy_signal = price.copy()
    else:
        noise_power = signal_power / (10**(snr_db/10))
        noise = np.random.normal(0, np.sqrt(noise_power), len(price))
        noisy_signal = price + noise
    
    ccf_noisy = compute_crosscorrelation(template, noisy_signal, normalized=True)
    det_pos_n = np.argmax(ccf_noisy)
    loc_error_n = abs(det_pos_n - template_start)
    peak_n = ccf_noisy[det_pos_n]
    
    robustness_results.append({
        'Condition': nlabel,
        'Detected Position': det_pos_n,
        'Localization Error (days)': loc_error_n,
        'Peak NCC': peak_n
    })
    print(f"  {nlabel}: detected at {det_pos_n}, error = {loc_error_n} days, peak = {peak_n:.4f}")

# Amplitude scaling robustness
print("\nCROSS-CORRELATION ROBUSTNESS: Amplitude Scaling")
print("-" * 70)
for scale in [1.0, 0.8, 1.2]:
    scaled_template = template * scale
    ccf_scaled = compute_crosscorrelation(scaled_template, price, normalized=True)
    det_pos_s = np.argmax(ccf_scaled)
    print(f"  Scale = {scale}: detected at {det_pos_s}, "
          f"error = {abs(det_pos_s - template_start)} days, "
          f"peak = {ccf_scaled[det_pos_s]:.4f}")

print("=" * 70)

CROSS-CORRELATION ROBUSTNESS: Noise Perturbation
  No noise: detected at 50, error = 0 days, peak = 1.0000
  SNR=20dB: detected at 50, error = 0 days, peak = 0.9394
  SNR=10dB: detected at 4, error = 46 days, peak = 0.8916

CROSS-CORRELATION ROBUSTNESS: Amplitude Scaling
----------------------------------------------------------------------
  Scale = 1.0: detected at 50, error = 0 days, peak = 1.0000
  Scale = 0.8: detected at 50, error = 0 days, peak = 1.0000
  Scale = 1.2: detected at 50, error = 0 days, peak = 1.0000


In [8]:
# Summary comparison table
print("\n" + "=" * 80)
print("EVALUATION SUMMARY TABLE")
print("=" * 80)

rob_df = pd.DataFrame(robustness_results)
print("\nRobustness Analysis (Noise):")
print(rob_df.to_markdown(index=False, floatfmt='.4f'))

print("\nTemplate Length Comparison:")
tl_data = [{'Template Length': tl, 'Detected Pos': r['det_pos'],
            'Loc. Error (days)': r['loc_error'], 'Peak NCC': r['peak_val']}
           for tl, r in ccf_results.items()]
tl_df = pd.DataFrame(tl_data)
print(tl_df.to_markdown(index=False, floatfmt='.4f'))
print("=" * 80)


EVALUATION SUMMARY TABLE

Robustness Analysis (Noise):
| Condition   |   Detected Position |   Localization Error (days) |   Peak NCC |
|:------------|--------------------:|----------------------------:|-----------:|
| No noise    |                  50 |                           0 |     1.0000 |
| SNR=20dB    |                  50 |                           0 |     0.9394 |
| SNR=10dB    |                   4 |                          46 |     0.8916 |

Template Length Comparison:
|   Template Length |   Detected Pos |   Loc. Error (days) |   Peak NCC |
|------------------:|---------------:|--------------------:|-----------:|
|            5.0000 |        50.0000 |              0.0000 |     1.0000 |
|           10.0000 |        50.0000 |              0.0000 |     1.0000 |
|           20.0000 |        50.0000 |              0.0000 |     1.0000 |
